# Basic Setup

In [ ]:
import json
import pandas as pd
from pathlib import Path
import operator

identity = ["anonymous", "real_full_identity"]
datasets = ["mmlu", "moralchoice", "MoralMachineHuman"]

identity_revealed_folders = [Path(".").resolve()/identity[1]/x for x in datasets]
anonymous_folders = [Path(".").resolve()/identity[0]/x for x in datasets]

rich_family = ["12B", "32B", "70B"]

agent_family = ["Qwen", "Gemma", "LLaMA", "Mistral"]

result_base_folder = "./results"

experiments = ["experiment_" + str(x) for x in range(1,13)]
experiment_path = [result_base_folder+"/"+x for x in experiments]

In [ ]:
class AutoDict(dict):
	def __missing__(self, key):
		value = self[key] = self.__class__()
		return value

# Experiment 5

In [ ]:
experiment_5_full_data = AutoDict()

total_doi = 0

for anonymous, identity_revealed in zip(anonymous_folders, identity_revealed_folders):
	data_set_doi = 0
	data_set_array = []
	if not anonymous.exists():
		print(f"Directory not found: {anonymous.name}")
		continue
	if not identity_revealed.exists():
		print(f"Directory not found: {identity_revealed.name}")
		continue
	for anonymous_json, identity_json in zip(anonymous.rglob("*.json"), identity_revealed.rglob("*.json")):
		with anonymous_json.open("r", encoding="utf-8") as aj:
			content_aj = json.load(aj)
		with identity_json.open("r", encoding="utf-8") as ij:
			content_ij = json.load(ij)
		if (content_aj["simulation_metadata"]["pid"] != content_ij["simulation_metadata"]["pid"]):
			continue
		dataset = content_aj["simulation_metadata"]["dataset"].split("/")[1]
		pid = content_aj["simulation_metadata"]["pid"]
		agents_ij = content_ij["summary_analytics"]["agent_trajectories"]
		agents_aj = content_aj["summary_analytics"]["agent_trajectories"]
		agent_ij_aj_pair = {ayn["agent_id"]: {"anonymous":ayn["vote_history"], "identity":idr["vote_history"]} for ayn in agents_ij for idr in agents_aj if ayn["agent_id"] == idr["agent_id"]}
		doi = 0
		for agent in agent_ij_aj_pair.values():
			for ayn_vote, idr_vote, ayn_prev, idr_prev in zip(agent["anonymous"][1:], agent["identity"][1:], agent["anonymous"][:14], agent["identity"][:14]):
				flip_ayn = True if ayn_prev != ayn_vote else False
				flip_idr = True if idr_prev != idr_vote else False
				doi += (operator.xor(flip_ayn, flip_idr))
		final_data = {
			"dataset": dataset,
			"pid": pid,
			"agents identity": agents_ij,
			"agents anonymouys": agents_aj,
			"Degree of Identity": doi
		}
		data_set_array.append(final_data)
		data_set_doi+=doi
	dataset_details = {
		"Degree of Identity": data_set_doi
	}
	experiment_5_full_data[dataset]["Array"] = data_set_array
	experiment_5_full_data[dataset]["Dataset final"] = dataset_details
	total_doi += data_set_doi
experiment_5_full_data["Final results"] = {
	"Degree of Identity": total_doi
}

with open(experiment_path[4]+"/results.json", "w", encoding="utf-8") as f:
	json.dump(experiment_5_full_data, f, indent=4)

In [ ]:
excel_data = AutoDict()
# excel_data["mmlu"] = None
# excel_data["moralchoice"] = None
# excel_data["MoralMachineHuman"] = None

excel_data["Master analytics"] = {
	"mmlu": experiment_5_full_data["mmlu"]["Dataset final"],
	"moralchoice": experiment_5_full_data["moralchoice"]["Dataset final"],
	"MoralMachineHuman": experiment_5_full_data["MoralMachineHuman"]["Dataset final"],
	"Final Results": experiment_5_full_data["Final results"]
}

excel_path = Path(experiment_path[4]+"/result.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
	for sheets, records in excel_data.items():
		df = pd.DataFrame.from_dict(records, orient="index")
		df.index.name = "Dataset"
		df.reset_index(inplace=True)
		df.to_excel(writer, sheet_name=sheets, index=False)
excel_data